# Tutorial 24: Agent Middleware in LangChain 1.0

In this tutorial we explore the middleware system introduced in LangChain 1.0 (September 2025). Middleware lets you inject logic at any point in the agent loop — before the model call, after it, or to modify the request itself. It is the foundation of context engineering.

## 1. What Is Middleware?

In web frameworks, middleware intercepts requests and responses. In LangChain 1.0, middleware intercepts the agent's LLM calls:

```
Agent loop:
  ... → [before_model] → [wrap_model_call] → LLM → [after_model] → ...
```

| Hook | When | Common uses |
|---|---|---|
| `before_model` | Before every LLM call | Logging, routing, state checks, memory injection |
| `after_model` | After every LLM call | Guardrails, HITL, output validation |
| `wrap_model_call` | Wraps the actual API call | Edit system prompt, messages, tools, retry logic |

Middleware is defined as classes that subclass `AgentMiddleware` and override the relevant methods.

## 2. Setup

In [1]:
import os
import time
from datetime import datetime
from langchain.agents import create_agent
from langchain.agents.middleware.types import AgentMiddleware
from langchain.agents.middleware.human_in_the_loop import HumanInTheLoopMiddleware
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

model = init_chat_model("groq:qwen/qwen3-32b", temperature=0.1)

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 3. Defining Tools for the Agent

In [2]:
@tool
def search_database(query: str) -> str:
    """Search the company database for information."""
    return f"Database results for '{query}': Found 3 matching records. Top result: Customer #1042, Revenue: $15,200."


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    return f"Email sent to {to} with subject '{subject}'."


@tool
def update_record(record_id: str, field: str, value: str) -> str:
    """Update a field in a database record."""
    return f"Record {record_id} updated: {field} = {value}."


tools = [search_database, send_email, update_record]
print(f"Defined {len(tools)} tools.")

Defined 3 tools.


## 4. Writing a Logging Middleware

Middleware is implemented as a class that subclasses `AgentMiddleware`. Override `before_model` and/or `after_model` to add logic around every LLM call. Both methods receive the current agent `state` and a `runtime` context.

In [3]:
call_log = []  # in production this would write to a database or logging service


class LoggingMiddleware(AgentMiddleware):
    """Logs every LLM call with message count and response length."""

    def before_model(self, state, runtime):
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event": "before_model",
            "message_count": len(state.get("messages", [])),
        }
        call_log.append(entry)
        print(f"  [LOG] Before model call | messages: {entry['message_count']}")

    def after_model(self, state, runtime):
        last_msg = state.get("messages", [{}])[-1]
        content = getattr(last_msg, 'content', '') or ''
        entry = {
            "timestamp": datetime.now().isoformat(),
            "event": "after_model",
            "response_length": len(content),
        }
        call_log.append(entry)
        print(f"  [LOG] After model call  | response length: {entry['response_length']}")


print("LoggingMiddleware defined.")

LoggingMiddleware defined.


## 5. Writing a Guardrail Middleware

A guardrail middleware uses `after_model` to validate the LLM's output before the agent acts on it.

In [4]:
BLOCKED_TOPICS = ["competitor", "salary", "confidential", "lawsuit"]


class GuardrailMiddleware(AgentMiddleware):
    """Blocks responses containing sensitive topics by replacing them with a safe fallback."""

    def after_model(self, state, runtime):
        messages = state.get("messages", [])
        if not messages:
            return None

        last_msg = messages[-1]
        content = (getattr(last_msg, 'content', '') or '').lower()

        for topic in BLOCKED_TOPICS:
            if topic in content:
                print(f"  [GUARDRAIL] Blocked — response mentions '{topic}'")
                safe_response = AIMessage(
                    content="I can't discuss that topic. Is there anything else I can help you with?"
                )
                return {"messages": messages[:-1] + [safe_response]}

        return None


print("GuardrailMiddleware defined.")

GuardrailMiddleware defined.


## 6. Writing a Context Injection Middleware

`wrap_model_call` wraps the entire LLM call — you receive the `request` (which carries `messages`, `system_message`, `tools`, etc.) and a `handler` callable that actually runs the model. Call `request.override(...)` to produce a modified copy, then pass it to `handler`.

> **Important**: always **prepend** your context to the existing system message rather than replacing it. `create_agent()` injects its own system message with tool-use instructions; overwriting it removes those instructions and causes the agent to loop forever.

In [5]:
class ContextInjectionMiddleware(AgentMiddleware):
    """Prepends dynamic context (date, user role) to every model request's system message."""

    def wrap_model_call(self, request, handler):
        date_prefix = f"Current date: {datetime.now().strftime('%Y-%m-%d')}."

        # Prepend to existing system message — never replace it entirely.
        # create_agent() injects tool-use instructions into the system message;
        # replacing it would remove the stop condition and cause an infinite loop.
        existing = request.system_message.content if request.system_message else ""
        new_content = f"{date_prefix}\n\n{existing}" if existing else date_prefix

        new_request = request.override(system_message=SystemMessage(content=new_content))
        print(f"  [CONTEXT] Injected date prefix into system message.")
        return handler(new_request)


print("ContextInjectionMiddleware defined.")

ContextInjectionMiddleware defined.


## 7. Composing Middleware with `create_agent()`

`create_agent()` is the LangChain 1.0 agent constructor. Pass middleware as a list of **class instances** (not functions). Multiple middleware run in order — first in the list is outermost.

In [6]:
agent = create_agent(
    model=model,
    tools=tools,
    middleware=[
        LoggingMiddleware(),
        GuardrailMiddleware(),
        ContextInjectionMiddleware(),
    ]
)

print("Agent with middleware created.")

Agent with middleware created.


In [7]:
config = {"configurable": {"thread_id": "middleware-demo", "user_role": "admin"}}

print("=== NORMAL REQUEST ===")
result = agent.invoke(
    {"messages": [HumanMessage(content="Search for information about customer 1042.")]},
    config
)
print("Response:", result["messages"][-1].content[:150])

=== NORMAL REQUEST ===
  [LOG] Before model call | messages: 1
  [CONTEXT] Injected date prefix into system message.


  [LOG] After model call  | response length: 0
  [LOG] Before model call | messages: 3
  [CONTEXT] Injected date prefix into system message.
  [LOG] After model call  | response length: 240
Response: Here's the information I found for customer 1042:

**Customer #1042**  
Revenue: $15,200  

There are 2 additional matching records in the database. W


In [8]:
print("=== BLOCKED REQUEST ===")
result_blocked = agent.invoke(
    {"messages": [HumanMessage(content="Tell me about our confidential salary data.")]},
    {"configurable": {"thread_id": "middleware-demo-2"}}
)
print("Response:", result_blocked["messages"][-1].content)

=== BLOCKED REQUEST ===
  [LOG] Before model call | messages: 1
  [CONTEXT] Injected date prefix into system message.
  [LOG] After model call  | response length: 0
  [LOG] Before model call | messages: 3
  [CONTEXT] Injected date prefix into system message.
  [GUARDRAIL] Blocked — response mentions 'salary'
  [LOG] After model call  | response length: 71
Response: I can't discuss that topic. Is there anything else I can help you with?


## 8. HITL Middleware

`HumanInTheLoopMiddleware` is a built-in middleware that uses `after_model` to intercept specific tool calls and call `interrupt()`, pausing graph execution until a human responds. Pass `interrupt_on` with the tool names to guard.

When `invoke()` hits an interrupt it returns normally, with an `__interrupt__` key in the result containing the pending decision details. Resume by calling `invoke(Command(resume={...}), config)` with the human's decision.

In [9]:
# create_agent() supports checkpointer directly — no separate .compile() needed
hitl_agent = create_agent(
    model=model,
    tools=tools,
    middleware=[
        LoggingMiddleware(),
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True, "update_record": True}
        ),
    ],
    checkpointer=MemorySaver()
)

print("HITL agent created.")

HITL agent created.


In [10]:
hitl_config = {"configurable": {"thread_id": "hitl-demo"}}

# invoke() returns normally when interrupted — the __interrupt__ key carries the pending review
result = hitl_agent.invoke(
    {"messages": [HumanMessage(content="Send a welcome email to alice@example.com about our new product.")]},
    hitl_config
)

if "__interrupt__" in result:
    interrupt_data = result["__interrupt__"][0].value
    print("Graph paused — awaiting approval.")
    print(f"Pending action: {interrupt_data['action_requests'][0]['name']}")
    print(f"Args: {interrupt_data['action_requests'][0]['args']}")
else:
    print("Completed without interrupt:", result["messages"][-1].content[:100])

  [LOG] Before model call | messages: 1
Graph paused — awaiting approval.
Pending action: send_email
Args: {'body': "Dear Alice, we're excited to welcome you to our new product. Discover the features designed to enhance your experience. Best regards, [Your Company]", 'subject': 'Welcome to Our New Product!', 'to': 'alice@example.com'}


In [11]:
# Resume with an "approve" decision — the format matches HITLRequest's decisions schema
result = hitl_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    hitl_config
)
print("Final response:", result["messages"][-1].content[:200])

  [LOG] After model call  | response length: 0
  [LOG] Before model call | messages: 3
  [LOG] After model call  | response length: 199
Final response: The welcome email has been successfully sent to alice@example.com with the subject "Welcome to Our New Product!". Let me know if you'd like to follow up with additional actions or send more emails! 🚀


## 9. Viewing the Call Log

In [12]:
print(f"Logged {len(call_log)} events:")
for entry in call_log[-6:]:
    print(f"  {entry['timestamp']} | {entry['event']} | {dict(list(entry.items())[2:])}")

Logged 12 events:
  2026-05-06T17:28:24.714233 | before_model | {'message_count': 3}
  2026-05-06T17:28:25.613581 | after_model | {'response_length': 71}
  2026-05-06T17:28:25.644317 | before_model | {'message_count': 1}
  2026-05-06T17:28:26.201929 | after_model | {'response_length': 0}
  2026-05-06T17:28:26.206151 | before_model | {'message_count': 3}
  2026-05-06T17:28:26.799215 | after_model | {'response_length': 199}


## 10. Conclusion

In this tutorial we built and composed Agent Middleware using LangChain 1.0:
- `AgentMiddleware` subclass — the correct way to define middleware (not plain functions)
- `before_model` — runs before every LLM call (logging, routing)
- `after_model` — runs after every LLM call (guardrails, HITL, validation)
- `wrap_model_call(request, handler)` — wraps the actual API call; use `request.override(...)` to modify messages, system prompt, or tools
- `HumanInTheLoopMiddleware(interrupt_on={...})` — built-in HITL middleware using `interrupt()`
- `create_agent(model, tools, middleware=[...], checkpointer=...)` — pass middleware instances and checkpointer directly

In Tutorial 25, the final tutorial, we explore MCP and A2A — the open protocols that let LangGraph agents communicate with standardised tool servers and other AI frameworks.